In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import xarray as xr

from ml_ds.network import LightningModule
from ml_ds.train_CNN import (
    INPUT_FILES,
    INPUT_VARS,
    STATIC_VARS,
    TARGET_VARS,
    initialize_model,
    load_datasets,
)

# CRITERION = nn.MSELoss()
CRITERION = nn.L1Loss()

In [ ]:
def to_xarray(tensor, vars, coords):
    tensor = tensor.detach().numpy()
    return xr.Dataset(
        {
            var: xr.DataArray(tensor[:, var_num, :, :], coords=coords)
            for var_num, var in enumerate(vars)
        }
    )

In [ ]:
# Put test result back into xarray with coordinates.
era5_coords = xr.open_dataset(INPUT_FILES[0])
era5_coords = {
    "valid_time": era5_coords["valid_time"].values[:1],
    "latitude": era5_coords["latitude"].values,
    "longitude": era5_coords["longitude"].values,
}

In [ ]:
train_data, val_data, test_data = load_datasets()
model = initialize_model()

In [ ]:
network = LightningModule.load_from_checkpoint(
    "../checkpoints/last.ckpt",
    map_location="cpu",
    model=model,
    train_dataset=train_data,
    val_dataset=val_data,
    test_dataset=test_data,
    batch_size=1, 
    num_workers=1,
    weights_only=False,
)

In [ ]:
test_loader = iter(network.test_dataloader())

In [ ]:
x, y = next(test_loader)
yh = network.forward(x)
# x = train_data.input_means + x * train_data.input_sds
# y = train_data.target_means + y * train_data.target_sds
# yh = train_data.target_means + yh * train_data.target_sds

In [ ]:
CRITERION(yh, y)

In [ ]:
x = to_xarray(x, INPUT_VARS + STATIC_VARS, era5_coords)
y = to_xarray(y, TARGET_VARS, era5_coords)
yh = to_xarray(yh, TARGET_VARS, era5_coords)

In [ ]:
plot_var = "u10"
time_idx = 0

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 5))

x[plot_var][time_idx].plot.imshow(ax=ax[0])
y[plot_var][time_idx].plot.imshow(ax=ax[1])
yh[plot_var][time_idx].plot.imshow(ax=ax[2])

ax[0].set_title("x: input")
ax[1].set_title("y: truth")
ax[2].set_title("yh: prediction")

In [ ]:
loss_yx = CRITERION(torch.tensor(y[plot_var].values), torch.tensor(x[plot_var].values))
print(f"Loss between y and x: {loss_yx.item()}")
yx_diff = y[plot_var] - x[plot_var]
print(f"Diff sum: {yx_diff.sum().values}")

In [ ]:
loss_yyh = CRITERION(torch.tensor(y[plot_var].values), torch.tensor(yh[plot_var].values))
print(f"Loss between y and yh: {loss_yyh.item()}")
yyh_diff = y[plot_var] - yh[plot_var]
print(f"Diff sum: {yyh_diff.sum().values}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

yx_diff[time_idx].plot.imshow(ax=ax[0])
yyh_diff[time_idx].plot.imshow(ax=ax[1])

ax[0].set_title("y - x: difference")
ax[1].set_title("y - yh: difference")